In [13]:
import pandas as pd

fear = pd.read_csv("fear_greed_index.csv")
trades = pd.read_csv("historical_data.csv")

print(fear.shape)
print(trades.shape)

print(fear.isnull().sum())
print(trades.isnull().sum())

print(fear.duplicated().sum())
print(trades.duplicated().sum())

fear['date'] = pd.to_datetime(fear['date']).dt.date
trades['Timestamp'] = pd.to_datetime(trades['Timestamp'], unit='ms')
trades['date'] = trades['Timestamp'].dt.date

merged = trades.merge(
    fear[['date', 'classification']],
    on='date',
    how='inner'
)

print(merged.head())

merged['is_win'] = merged['Closed PnL'] > 0
merged['is_long'] = merged['Direction'].str.lower() == 'buy'

daily = merged.groupby(['date', 'Account', 'classification']).agg(
    daily_pnl=('Closed PnL', 'sum'),
    trades_per_day=('Closed PnL', 'count'),
    win_rate=('is_win', 'mean'),
    avg_trade_size=('Size USD', 'mean'),
    long_ratio=('is_long', 'mean')
).reset_index()

(2644, 4)
(211224, 16)
timestamp         0
value             0
classification    0
date              0
dtype: int64
Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64
0
0
                                      Account  Coin  Execution Price  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9769   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9800   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9855   
3  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9874   
4  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9894   

   Size Tokens  Size USD Side     Timestamp IST  Start Position Direction  \
0    

In [15]:
daily.groupby('classification')['daily_pnl'].mean()

classification
Extreme Greed     35393.098355
Fear             209372.662205
Greed             99675.516731
Neutral           19842.797260
Name: daily_pnl, dtype: float64

In [17]:
daily.groupby('classification')['win_rate'].mean()

classification
Extreme Greed    0.336609
Fear             0.415878
Greed            0.374074
Neutral          0.260683
Name: win_rate, dtype: float64

In [19]:
daily.groupby('classification')['trades_per_day'].mean()

classification
Extreme Greed    1392.40000
Fear             4183.46875
Greed            1134.03125
Neutral           892.62500
Name: trades_per_day, dtype: float64

In [21]:
daily.groupby('classification')['long_ratio'].mean()

classification
Extreme Greed    0.200000
Fear             0.082953
Greed            0.186593
Neutral          0.152501
Name: long_ratio, dtype: float64

In [23]:
daily.groupby('classification')['avg_trade_size'].mean()

classification
Extreme Greed    4344.447836
Fear             5926.522723
Greed            5839.310974
Neutral          3793.444161
Name: avg_trade_size, dtype: float64

In [25]:
activity = daily.groupby('Account')['trades_per_day'].mean()
median_value = activity.median()

daily['segment'] = daily['Account'].apply(
    lambda x: 'High Frequency' if activity[x] > median_value else 'Low Frequency'
)

daily.groupby(['classification','segment'])['daily_pnl'].mean()

classification  segment       
Extreme Greed   High Frequency     60843.169305
                Low Frequency      -2782.008070
Fear            High Frequency    324428.018469
                Low Frequency      94317.305942
Greed           High Frequency    158438.794314
                Low Frequency      47825.565921
Neutral         High Frequency      6867.420164
                Low Frequency      32818.174355
Name: daily_pnl, dtype: float64

In [27]:
consistency = daily.groupby('Account')['daily_pnl'].mean()

median_profit = consistency.median()

daily['profit_segment'] = daily['Account'].apply(
    lambda x: 'Consistent Winner' if consistency[x] > median_profit else 'Inconsistent'
)

daily.groupby(['classification','profit_segment'])['daily_pnl'].mean()

classification  profit_segment   
Extreme Greed   Consistent Winner     45632.376978
                Inconsistent          -5564.016140
Fear            Consistent Winner    394276.467484
                Inconsistent          24468.856927
Greed           Consistent Winner    170628.619440
                Inconsistent          28722.414021
Neutral         Consistent Winner     45539.571589
                Inconsistent          -5853.977069
Name: daily_pnl, dtype: float64

In [29]:
size_avg = daily.groupby('Account')['avg_trade_size'].mean()
size_median = size_avg.median()

daily['size_segment'] = daily['Account'].apply(
    lambda x: 'Large Trader' if size_avg[x] > size_median else 'Small Trader'
)

daily.groupby(['classification','size_segment'])['daily_pnl'].mean()

classification  size_segment
Extreme Greed   Large Trader     -3625.530805
                Small Trader     93921.042095
Fear            Large Trader    278058.006946
                Small Trader    140687.317465
Greed           Large Trader    100204.801639
                Small Trader     99075.660501
Neutral         Large Trader      9202.357515
                Small Trader     30483.237005
Name: daily_pnl, dtype: float64